In [1]:
import numpy as np
import pandas as pd
import  tpqoa
import time
from datetime import datetime, timedelta, timezone
import warnings
warnings.filterwarnings('ignore')

In [30]:
class ConTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, window, units):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.data = None
        self.last_bar = None
        self.units = units
        self.position = 0
        self.profits = [] # NEW

        #*****************add strategy-specific attributes here******************
        self.window = window
        #************************************************************************

    def get_most_recent(self, days = 5):
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        now = now - timedelta(microseconds = now.microsecond)
        past = now - timedelta(days = days)
        df = self.get_history(instrument = self.instrument, start = past, end = now,
                               granularity = "S5", price = "M", localize = False).c.dropna().to_frame()
        df.rename(columns = {"c":self.instrument}, inplace = True)
        df = df.resample(self.bar_length, label = "right").last().dropna().iloc[:-1]
        self.raw_data = df.copy()
        self.last_bar = self.raw_data.index[-1]

    def on_success(self, time, bid, ask):
        print(self.ticks, end = " ")

        # collect and store tick data
        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument:(ask + bid)/2},
                          index = [recent_tick])
        self.tick_data = pd.concat([self.tick_data, df]) # new with pd.concat()

        # if a time longer than the bar_lenght has elapsed between last full bar and the most recent tick
        if recent_tick - self.last_bar >= self.bar_length:
            self.resample_and_join()
            self. moving_average()
            self.execute_trades()

    def resample_and_join(self):
        self.raw_data = pd.concat([self.raw_data, self.tick_data.resample(self.bar_length,
                                                                          label="right").last().ffill().iloc[:-1]])
        self.tick_data = self.tick_data.iloc[-1:]
        self.last_bar = self.raw_data.index[-1]

    # def define_strategy(self): # "strategy-specific"
    #     df = self.raw_data.copy()
    #
    #     #******************** define your strategy here ************************
    #     df["returns"] = np.log(df[self.instrument] / df[self.instrument].shift(1))
    #     df["position"] = np.sign(df.returns.rolling(self.window).mean())
    #     #***********************************************************************
    #
    #     self.data = df.copy()

    def moving_average(self):
        df = self.raw_data.copy()
        sma_l = 20
        sma_h = 50
        df['SMA_20'] = df[self.instrument].rolling(20).mean()
        df['SMA_50'] = df[self.instrument].rolling(50).mean()
        df['position'] = np.where(df['SMA_20'] > df['SMA_50'], 1, -1)

        self.data = df.copy()

    def execute_trades(self):
        if self.data["position"].iloc[-1] == 1:
            if self.position == 0:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            elif self.position == -1:
                order = self.create_order(self.instrument, self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING LONG")  # NEW
            self.position = 1
        elif self.data["position"].iloc[-1] == -1:
            if self.position == 0:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units * 2, suppress = True, ret = True)
                self.report_trade(order, "GOING SHORT")  # NEW
            self.position = -1
        elif self.data["position"].iloc[-1] == 0:
            if self.position == -1:
                order = self.create_order(self.instrument, self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            elif self.position == 1:
                order = self.create_order(self.instrument, -self.units, suppress = True, ret = True)
                self.report_trade(order, "GOING NEUTRAL")  # NEW
            self.position = 0

    def report_trade(self, order, going):  # NEW
        time = order["time"]
        units = order["units"]
        price = order["price"]
        pl = float(order["pl"])
        self.profits.append(pl)
        cumpl = sum(self.profits)
        print("\n" + 100* "-")
        print("{} | {}".format(time, going))
        print("{} | units = {} | price = {} | P&L = {} | Cum P&L = {}".format(time, units, price, pl, cumpl))
        print(100 * "-" + "\n")


In [31]:
# def moving_average(self):
    #     df = self.raw_data.copy()
    #     sma_l = 20
    #     sma_h = 50
    #     df['SMA_20'] = df[self.instrument].rolling(20).mean()
    #     df['SMA_50'] = df[self.instrument].rolling(50).mean()
    #     df['position'] = np.where(df['SMA_20'] > df['SMA_50'], 1, -1)
    #
    #     self.data = df.copy()

In [36]:
trader = ConTrader('oanda.cfg', 'EUR_USD', bar_length='5s', window=1, units=100000)

In [42]:
trader.get_most_recent()
trader.stream_data(trader.instrument, stop= 200)

if trader.position!= 0:
    close_order= trader.create_order(trader.instrument, units=-trader.position * trader.units, suppress=True, ret=True)
    trader.report_trade(close_order, "GOING NEUTRAL")
    trader.position=0

1 
----------------------------------------------------------------------------------------------------
2026-02-26T00:10:59.781931569Z | GOING LONG
2026-02-26T00:10:59.781931569Z | units = 100000.0 | price = 1.18136 | P&L = 0.0 | Cum P&L = 15.0
----------------------------------------------------------------------------------------------------

2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 
----------------------------------------------------------------------------------------------------
2026-02-26T00:14:53.850448922Z | GOING SHORT
2026-02-26T00:14:53.850448922Z | units = -200000.0 | price = 1.18129 | P&L = -7.0 | Cum P&L = 8.0
----------------------------------------------------------------------------------------------------

99 100 101 102 103 104 

In [43]:
trader.data

,EUR_USD,SMA_20,SMA_50,position
2026-02-20 17:54:00+00:00,1.178280,NaN,NaN,-1
2026-02-20 17:54:05+00:00,1.178350,NaN,NaN,-1
2026-02-20 17:54:10+00:00,1.178340,NaN,NaN,-1
2026-02-20 17:54:15+00:00,1.178340,NaN,NaN,-1
2026-02-20 17:54:20+00:00,1.178360,NaN,NaN,-1
...,...,...,...,...
2026-02-26 00:17:40+00:00,1.181435,1.181409,1.181370,1
2026-02-26 00:17:45+00:00,1.181435,1.181409,1.181372,1
2026-02-26 00:17:50+00:00,1.181435,1.181409,1.181374,1
2026-02-26 00:17:55+00:00,1.181475,1.181409,1.181377,1


In [29]:
trader.tick_data

,EUR_USD
2026-02-26 00:02:35.293344589+00:00,1.181475
2026-02-26 00:02:35.592369081+00:00,1.181480
2026-02-26 00:02:35.636361435+00:00,1.181480
2026-02-26 00:02:36.361205373+00:00,1.181465
